# A traditional event-loop analysis

Before introducing RDataFrame, it is useful to look at the traditional analysis pattern.

A typical event loop does the following:

1. Open a dataset.
2. Loop over events.
3. Read branch values for the current event.
4. Compute some quantity.
5. Apply selections.
6. Fill histograms.

This is a natural way to think about an analysis, but for large datasets it can become verbose and hard to optimize.

## Physics example: $\Upsilon \rightarrow \mu^{+}\mu^{-}$ with CMS open data

We will use a simple dimuon example.
Let's get started by exploring the CMS data.


## Setup

In [ ]:
import math
import ROOT

## Open the dataset

A `TChain` behaves like one large `TTree`, even if the data are split across several ROOT files.
Here we only add one file, but the same pattern also works for many files.

In [ ]:
tree = ROOT.TChain("my_tree")
tree.Add("data/data_Upsilon.root")

n_events = tree.GetEntries()

print(f"Number of events: {n_events}")

## Inspect the tree

Before writing an analysis, it is useful to look at the structure of the dataset.

In [ ]:
tree.Print()

The kinematic properties of the muons are stored in branches. For the first muon they are called `Q1, E1, px1, py1, pz1, pt1, eta1, phi1,` and for the second muon they are called `Q2, E2, px2, py2, pz2, pt2, eta2, phi2`.


**Muon Info**:
|Variable|Description|
| -------- | ------- |
|**type** |	Muon type: whether the muon is a tracker (T) or global muon (G)|
|**E** |	The total energy of the muon (GeV)|
|**px,py,pz**| 	The components of the momemtum of the muon (GeV)|
|**pt** |	The transverse momentum of the muon (GeV)|
|**eta** |	The pseudorapidity of the muon|
|**phi** |	The phi angle of the muon (rad)|
|**Q** |	The charge of the muon|

We can also print the content of a single event.

In [ ]:
tree.Show(10)

Each event contains two reconstructed muons. For each muon, the dataset stores its
energy and momentum components:

- `E1`, `px1`, `py1`, `pz1` for the first muon,
- `E2`, `px2`, `py2`, `pz2` for the second muon.

From these quantities we can compute the invariant mass of the two-muon system.
A histogram of this mass can reveal resonances such as the $\Upsilon$ family.

The invariant mass of two particles is computed from the total energy and total momentum:
m^2 = E^2 - p_x^2 - p_y^2 - p_z^2 .

In [ ]:
def invariant_mass(px1, py1, pz1, E1, px2, py2, pz2, E2):
    """Invariant mass of a two-particle system."""
    ### YOUR CODE

## Book a histogram

We want to fill a histogram with the dimuon invariant mass.

The histogram definition contains:

- an internal object name: `"hist_loop"`,
- a title and axis labels,
- the number of bins,
- the lower and upper edge of the histogram.

In [ ]:
x_min = 8.0
x_max = 12.0
n_bins = 200

x_title = "M_{#mu#mu} (GeV)"
y_title = "Events"
title = "Dimuon mass"

hist_loop = ROOT.TH1F(
    "hist_loop",
    f"{title};{x_title};{y_title}",
    n_bins,
    x_min,
    x_max,
)

## The event loop

This is the traditional analysis pattern:

1. Loop over all events.
2. Load the current event with `GetEntry`.
3. Compute the dimuon invariant mass.
4. Apply a selection.
5. Fill the histogram.

For this example we keep only candidates in the mass range of interest.

In [ ]:
%%time

for i in range(n_events):
    tree.GetEntry(i)

    mass = invariant_mass(
        tree.px1, tree.py1, tree.pz1, tree.E1,
        tree.px2, tree.py2, tree.pz2, tree.E2,
    )

    # Simple mass-window selection
    if mass < 8.5 or mass > 11.5:
        continue
    hist_loop.Fill(mass)


## Draw the result

In [ ]:
%jsroot on

canvas = ROOT.TCanvas()
hist_loop.Draw("e1")
canvas.Draw()

## What did the event loop do?

The essential analysis logic was:

```python
for event in dataset:
    mass = compute_mass(event)

    if not selection(mass):
        continue

    histogram.Fill(mass)
```

This is a very natural way to think about data analysis.

However, for large analyses this style can become verbose:

- the loop is explicit,
- selections are mixed with calculations,
- intermediate quantities are not named as reusable columns,
- parallel execution is not automatic,
- the analysis flow can become difficult to read.

## Exercise for later

After learning the basic RDataFrame operations, come back to this example and rewrite the event-loop analysis.

Your RDataFrame version should:

1. Open the same tree.
2. Define a new column containing the dimuon mass.
3. Filter events in the mass window \(8.5 < M_{\mu\mu} < 11.5\) GeV.
4. Fill the same histogram.

This exercise is intentionally left for later: first we need to learn the RDataFrame syntax.

In [ ]:
# Exercise skeleton for later:
#
# df = ...
#
# histo = (
#     df
#     .Define(...)
#     .Filter(...)
#     .Histo1D(...)
# )